# SL-9 --- Resolution Inverse et Progol (ILP bottom-up)

**Navigation** : [Index](./README.md) | [<< SL-8](SL-8-ActiveAutomataLearning.ipynb) | [SL-10 >>](SL-10-Capstone-NeuroSymbolic.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Expliquer l'idee directrice de l'ILP bottom-up : **inverser la deduction**
2. Implementer la **generalisation la moins generale** (LGG) de Plotkin et constater son inflation
3. Definir et implementer la **theta-subsomption**, l'ordre de generalite du treillis des clauses
4. Construire la **clause bottom** d'un exemple (entailment inverse, Muggleton 1995)
5. Derouler la **recherche a la Progol** : explorer le treillis borne par la clause bottom
6. Situer FOIL (SL-4), Progol, et les systemes modernes (Popper, ILASP)

### Prerequis
- SL-4 (clauses de Horn, unification, FOIL, operateurs V/W)
- Python pur : aucune dependance externe dans ce notebook

### Duree estimee : 60 minutes


---

## 1. Inverser la deduction

SL-4 a presente les deux directions de l'ILP. **Top-down** (FOIL) : partir de la
clause la plus generale et la *specialiser* litteral par litteral, guide par un gain
statistique. **Bottom-up** : partir des exemples --- des clauses ultra-specifiques
--- et les *generaliser*. Les operateurs V (absorption) et W (identification) de
SL-4 en etaient un premier apercu : chacun inverse un pas de **resolution**, la
regle de deduction unique de la programmation logique. Si la resolution de $C_1$ et
$C_2$ produit $R$, l'apprentissage bottom-up se demande : *etant donnes $R$ (un
exemple) et $C_1$ (une connaissance de fond), quel $C_2$ (une regle) aurait permis
de le deduire ?*

Le probleme historique de cette idee : appliquee naivement, elle ne sait pas ou
s'arreter. Ce notebook suit la lignee qui l'a rendue praticable :

1. **Plotkin (1970)** : la generalisation la moins generale (LGG) --- generaliser
   *exactement le necessaire*, pas plus... mais les clauses produites enflent ;
2. **la theta-subsomption** : l'ordre de generalite qui structure l'espace des
   clauses en **treillis** ;
3. **Muggleton (1995), Progol** : l'**entailment inverse** --- construire pour
   chaque exemple une clause *bottom* $\bot(e)$ qui borne le treillis par le bas,
   puis chercher la meilleure clause *entre* $\top$ et $\bot(e)$.

Domaine fil rouge : l'arbre genealogique classique de Prolog, et la cible
`grandfather(X, Y)`.


---

## 2. Representation et couverture

Memes conventions qu'en SL-4, en plus compact : un **atome** est un tuple
`(predicat, arg1, arg2)`, une **variable** commence par une majuscule, une
**clause** est une paire `(tete, corps)`. La fonction centrale est `covers` :
une clause couvre un exemple si on peut unifier sa tete avec l'exemple puis
satisfaire chaque litteral du corps dans les faits de fond (backtracking).


In [1]:
# Representation : atomes = tuples, variables = chaines commencant par une majuscule
def is_var(t):
    return isinstance(t, str) and t[:1].isupper()

def subst(atom, theta):
    return (atom[0],) + tuple(theta.get(a, a) for a in atom[1:])

def match_atom(pattern, fact, theta):
    """Etend theta pour que pattern devienne fact ; None si impossible."""
    if pattern[0] != fact[0] or len(pattern) != len(fact):
        return None
    th = dict(theta)
    for p, f in zip(pattern[1:], fact[1:]):
        if is_var(p):
            if p in th and th[p] != f:
                return None
            th[p] = f
        elif p != f:
            return None
    return th

def body_satisfiable(body, facts, theta):
    if not body:
        return True
    for fact in facts:
        th = match_atom(body[0], fact, theta)
        if th is not None and body_satisfiable(body[1:], facts, th):
            return True
    return False

def covers(clause, example, facts):
    head, body = clause
    th = match_atom(head, example, {})
    return th is not None and body_satisfiable(list(body), facts, th)

def show_clause(clause):
    head, body = clause
    fmt = lambda a: f"{a[0]}({', '.join(a[1:])})"
    return fmt(head) + (" :- " + ", ".join(fmt(b) for b in body) if body else "") + "."


# Connaissance de fond : un arbre genealogique
FACTS = [
    ("parent", "tom", "bob"), ("parent", "tom", "liz"),
    ("parent", "bob", "ann"), ("parent", "bob", "pat"),
    ("parent", "liz", "sue"), ("parent", "pat", "jim"),
    ("parent", "sue", "eve"),
    ("male", "tom"), ("male", "bob"), ("male", "jim"),
    ("female", "liz"), ("female", "ann"), ("female", "pat"),
    ("female", "sue"), ("female", "eve"),
]

POS = [("grandfather", "tom", "ann"), ("grandfather", "tom", "pat"),
       ("grandfather", "tom", "sue"), ("grandfather", "bob", "jim")]
NEG = [("grandfather", "liz", "eve"),   # grand-parent, mais femme
       ("grandfather", "liz", "sue"),   # parent direct
       ("grandfather", "bob", "ann"),   # pere, pas grand-pere
       ("grandfather", "pat", "jim"),   # mere
       ("grandfather", "tom", "bob"),   # pere
       ("grandfather", "ann", "jim")]   # aucun lien

# La cible (que l'algorithme devra retrouver seul) :
target = (("grandfather", "X", "Y"),
          [("male", "X"), ("parent", "X", "Z"), ("parent", "Z", "Y")])
print(f"Cible : {show_clause(target)}\n")
print(f"{'exemple':>24} | attendu | couvert")
print("-" * 46)
for e in POS + NEG:
    exp = "+" if e in POS else "-"
    print(f"{show_clause((e, []))[:-1]:>24} |    {exp}    |   {'oui' if covers(target, e, FACTS) else 'non'}")

Cible : grandfather(X, Y) :- male(X), parent(X, Z), parent(Z, Y).

                 exemple | attendu | couvert
----------------------------------------------
   grandfather(tom, ann) |    +    |   oui
   grandfather(tom, pat) |    +    |   oui
   grandfather(tom, sue) |    +    |   oui
   grandfather(bob, jim) |    +    |   oui
   grandfather(liz, eve) |    -    |   non
   grandfather(liz, sue) |    -    |   non
   grandfather(bob, ann) |    -    |   non
   grandfather(pat, jim) |    -    |   non
   grandfather(tom, bob) |    -    |   non
   grandfather(ann, jim) |    -    |   non


La clause cible couvre exactement les positifs : la representation et le moteur
de couverture sont valides. Tout l'enjeu est maintenant de la **decouvrir** a
partir des exemples, sans la connaitre.


---

## 3. La generalisation la moins generale (Plotkin, 1970)

Premier outil bottom-up : etant donnees deux clauses, construire la clause la plus
specifique qui les generalise toutes les deux. Sur les termes, c'est
l'**anti-unification** --- le dual exact de l'unification de SL-4 : deux constantes
differentes sont remplacees par une variable, *la meme paire de constantes donnant
toujours la meme variable*.

$$lgg(f(s_1, \ldots), f(t_1, \ldots)) = f(lgg(s_1, t_1), \ldots) \qquad
lgg(s, t) = V_{s,t} \text{ si } s \neq t$$

Sur les clauses, le LGG croise tous les litteraux compatibles des deux corps ---
et c'est la que l'inflation commence.


In [2]:
def lgg_term(t1, t2, mapping):
    if t1 == t2:
        return t1
    if (t1, t2) not in mapping:
        mapping[(t1, t2)] = f"V{len(mapping) + 1}"
    return mapping[(t1, t2)]

def lgg_atom(a1, a2, mapping):
    if a1[0] != a2[0] or len(a1) != len(a2):
        return None
    return (a1[0],) + tuple(lgg_term(x, y, mapping) for x, y in zip(a1[1:], a2[1:]))

def lgg_clause(c1, c2):
    mapping = {}
    head = lgg_atom(c1[0], c2[0], mapping)
    body = []
    for b1 in c1[1]:
        for b2 in c2[1]:
            g = lgg_atom(b1, b2, mapping)
            if g is not None and g not in body:
                body.append(g)
    return (head, body)


# Deux "explications" au sol de deux exemples positifs differents :
expl_1 = (("grandfather", "tom", "ann"),
          [("male", "tom"), ("parent", "tom", "bob"), ("parent", "bob", "ann")])
expl_2 = (("grandfather", "bob", "jim"),
          [("male", "bob"), ("parent", "bob", "pat"), ("parent", "pat", "jim")])

g = lgg_clause(expl_1, expl_2)
print("LGG des deux explications :")
print(f"  {show_clause(g)}")
print(f"\nTaille des corps : {len(expl_1[1])} et {len(expl_2[1])} litteraux -> LGG : {len(g[1])} litteraux")
print(f"Couverture : {sum(covers(g, e, FACTS) for e in POS)}/4 positifs, "
      f"{sum(covers(g, e, FACTS) for e in NEG)}/6 negatifs")

LGG des deux explications :
  grandfather(V1, V2) :- male(V1), parent(V1, V3), parent(V4, V5), parent(bob, V6), parent(V3, V2).

Taille des corps : 3 et 3 litteraux -> LGG : 5 litteraux
Couverture : 4/4 positifs, 0/6 negatifs


### Interpretation : exact mais obese

Le LGG contient bien la cible --- les litteraux `male(V1)`, `parent(V1, V3)`,
`parent(V3, V2)` y figurent --- mais noyee parmi les produits croises : chaque
litteral `parent` de la premiere explication s'apparie avec *chacun* de ceux de la
seconde, engendrant des litteraux parasites avec des variables fraiches --- et notez
`parent(bob, V6)` : quand les deux explications partagent la *meme* constante,
le LGG la conserve telle quelle, sur-specialisant la clause a cet individu. Sur deux
clauses de 3 litteraux le degat est cosmetique ; mais le LGG de $n$ clauses peut
croitre **exponentiellement** en $n$, et la reduction du resultat (eliminer les
litteraux redondants sous subsomption, Plotkin encore) est elle-meme NP-difficile.

La generalisation pure ne suffit donc pas : il faut un *ordre* pour organiser
l'espace entre le trop-general et le trop-specifique --- c'est la theta-subsomption
--- puis une *borne* pour le tronquer --- ce sera la clause bottom.


---

## 4. La theta-subsomption : l'ordre du treillis

Une clause $C$ **theta-subsume** $D$ (note $C \succeq D$) s'il existe une
substitution $\theta$ telle que $C\theta \subseteq D$ (les litteraux de $C$
instancies sont tous dans $D$). C'est *l'ordre de generalite* de l'ILP :

- $C \succeq D$ implique $C \models D$ (si $C$ subsume $D$, $C$ implique $D$) ;
- la reciproque est **fausse** en general (les clauses recursives s'impliquent
  parfois sans se subsumer --- subtilite de Gottlob explores en exercice 4) ;
- contrairement a l'implication logique (indecidable entre clauses), la
  subsomption est decidable (mais NP-complete).

Sous cet ordre, les clauses forment un **treillis** : tout en haut la clause vide
de corps (qui subsume tout), tout en bas les exemples au sol. L'apprentissage
devient une *recherche dans le treillis*.


In [3]:
def subsumes(c, d):
    """C theta-subsume D ? On skolemise D (ses variables deviennent des
    constantes) puis on cherche theta plongeant les litteraux de C dans D."""
    d_vars = {v for atom in [d[0]] + list(d[1]) for v in atom[1:] if is_var(v)}
    sk = {v: f"sk_{v.lower()}" for v in d_vars}
    d_head, d_body = subst(d[0], sk), [subst(b, sk) for b in d[1]]

    theta0 = match_atom(c[0], d_head, {})
    if theta0 is None:
        return False
    def backtrack(lits, theta):
        if not lits:
            return True
        for t in d_body:
            th = match_atom(lits[0], t, theta)
            if th is not None and backtrack(lits[1:], th):
                return True
        return False
    return backtrack(list(c[1]), theta0)


top = (("grandfather", "X", "Y"), [])          # la clause la plus generale
ground = expl_1                                 # un exemple au sol (sect. 3)

chain = [("top", top), ("cible", target), ("exemple au sol", ground)]
print("L'ordre de generalite top >= cible >= exemple :")
for (n1, c1) in chain:
    for (n2, c2) in chain:
        if n1 != n2:
            print(f"  {n1:>15} subsume {n2:<15} ? {subsumes(c1, c2)}")

L'ordre de generalite top >= cible >= exemple :
              top subsume cible           ? True
              top subsume exemple au sol  ? True
            cible subsume top             ? False
            cible subsume exemple au sol  ? True
   exemple au sol subsume top             ? False
   exemple au sol subsume cible           ? False


### Lecture

La chaine est stricte : `top` subsume tout le monde, la cible subsume l'exemple au
sol (avec $\theta = \{X/tom, Z/bob, Y/ann\}$) mais pas l'inverse. C'est
exactement la structure dont un algorithme de recherche a besoin : descendre =
specialiser (ajouter des litteraux, FOIL), monter = generaliser (en retirer,
resolution inverse). Reste a borner la descente : c'est l'idee decisive de Progol.


---

## 5. L'entailment inverse : la clause bottom

L'observation de Muggleton (1995) : si une clause $C$ et la connaissance de fond
$B$ doivent expliquer l'exemple $e$ ($B \wedge C \models e$), alors par
contraposition $C$ doit subsumer une clause *calculable* : la **clause bottom**

$$\bot(e) = e \leftarrow \{\text{tous les faits de } B \text{ pertinents pour } e\}$$

ou « pertinents » signifie : atteignables depuis les constantes de $e$ en suivant
les faits, jusqu'a une profondeur bornee $i$ (la *profondeur de variables*). On
variabilise ensuite chaque constante. Toute hypothese candidate vit donc **entre
$\top$ et $\bot(e)$** dans le treillis : un espace fini, structure, enumerable.


In [4]:
def bottom_clause(example, facts, body_preds, depth=2):
    """Clause bottom de l'exemple : saturation bornee puis variabilisation."""
    in_scope = set(example[1:])
    ground_body = []
    for _ in range(depth):
        for f in facts:
            if f[0] in body_preds and f not in ground_body \
               and any(a in in_scope for a in f[1:]):
                ground_body.append(f)
        in_scope |= {a for f in ground_body for a in f[1:]}
    mapping = {}
    def var_of(const):
        if const not in mapping:
            mapping[const] = f"V{len(mapping) + 1}"
        return mapping[const]
    head = (example[0],) + tuple(var_of(c) for c in example[1:])
    body = [(f[0],) + tuple(var_of(a) for a in f[1:]) for f in ground_body]
    return (head, body), mapping


bottom, mapping = bottom_clause(POS[0], FACTS, {"parent", "male", "female"}, depth=2)
print(f"Exemple sature : {show_clause((POS[0], []))}")
print(f"\nClause bottom (profondeur 2, {len(bottom[1])} litteraux) :")
print(f"  {show_clause(bottom)}")
print(f"\nVariabilisation : " + ", ".join(f"{c} -> {v}" for c, v in mapping.items()))
print(f"\nLa cible subsume-t-elle la clause bottom ? {subsumes(target, bottom)}")

Exemple sature : grandfather(tom, ann).

Clause bottom (profondeur 2, 9 litteraux) :
  grandfather(V1, V2) :- parent(V1, V3), parent(V1, V4), parent(V3, V2), male(V1), female(V2), parent(V3, V5), parent(V4, V6), male(V3), female(V4).

Variabilisation : tom -> V1, ann -> V2, bob -> V3, liz -> V4, pat -> V5, sue -> V6

La cible subsume-t-elle la clause bottom ? True


### Interpretation

La clause bottom est la *version saturee* de l'exemple : tout ce que la connaissance
de fond sait dire sur `tom` et `ann` (et leurs voisins a distance 2) y figure,
variabilise. Et le theoreme se verifie : la cible subsume $\bot(e)$ --- ses trois
litteraux se plongent dans le corps de la clause bottom. *Toute* clause correcte
fera de meme : il suffit donc de chercher parmi les **sous-ensembles du corps de
$\bot(e)$**. La profondeur $i$ est le levier de ce compromis : trop petite, la
cible peut sortir de l'espace (exercice 2) ; trop grande, la clause bottom explose.


---

## 6. La recherche a la Progol

Progol explore le treillis $[\top, \bot(e)]$ du general au specifique (A* dans le
systeme original ; ici, une enumeration par taille croissante suffit). Chaque
candidat est un sous-ensemble *connecte* du corps de la clause bottom --- connecte :
chaque litteral doit etre relie a la tete par une chaine de variables partagees,
sinon il ne contraint rien. Le score est la **compression** $f = p - L$ sous
contrainte de consistance ($n = 0$) : couvrir beaucoup de positifs avec une clause
courte. Une boucle de **cover-set** (comme FOIL) retire les positifs couverts et
recommence tant qu'il en reste.


In [5]:
from itertools import combinations


def is_connected(head, body_subset):
    """Chaque litteral du corps doit etre relie a la tete par les variables."""
    reached = set(head[1:])
    pending = list(body_subset)
    progress = True
    while pending and progress:
        progress = False
        for lit in pending[:]:
            if set(lit[1:]) & reached:
                reached |= set(lit[1:])
                pending.remove(lit)
                progress = True
    return not pending


def progol_search(bottom, pos, neg, facts, max_len=3):
    """Meilleure clause consistante (n=0) dans [top, bottom], score = p - L."""
    head, blits = bottom
    best = None
    n_candidates = 0
    for k in range(1, max_len + 1):
        for sub in combinations(blits, k):
            if not is_connected(head, sub):
                continue
            n_candidates += 1
            cl = (head, list(sub))
            if any(covers(cl, e, facts) for e in neg):
                continue
            p = sum(covers(cl, e, facts) for e in pos)
            if p and (best is None or p - k > best[0]):
                best = (p - k, cl, p)
    return best, n_candidates


def progol_learn(pos, neg, facts, body_preds, depth=2, max_len=3):
    remaining = list(pos)
    theory = []
    while remaining:
        bottom, _ = bottom_clause(remaining[0], facts, body_preds, depth)
        best, n_cand = progol_search(bottom, remaining, neg, facts, max_len)
        if best is None:
            print(f"  echec sur {remaining[0]} : aucun candidat consistant")
            break
        score, clause, p = best
        print(f"  bottom = {len(bottom[1])} litteraux, {n_cand} candidats connectes"
              f" -> {show_clause(clause)}  (p={p}, score={score})")
        theory.append(clause)
        remaining = [e for e in remaining if not covers(clause, e, facts)]
    return theory


print("Apprentissage de grandfather/2 :")
theory = progol_learn(POS, NEG, FACTS, {"parent", "male", "female"})
print("\nTheorie finale :")
for c in theory:
    print(f"  {show_clause(c)}")
ok_pos = sum(covers(c, e, FACTS) for c in theory for e in POS)
ok_neg = sum(covers(c, e, FACTS) for c in theory for e in NEG)
print(f"\nVerification : {ok_pos}/{len(POS)} positifs couverts, {ok_neg}/{len(NEG)} negatifs couverts")

Apprentissage de grandfather/2 :
  bottom = 9 litteraux, 56 candidats connectes -> grandfather(V1, V2) :- parent(V1, V3), parent(V3, V2), male(V1).  (p=4, score=1)

Theorie finale :
  grandfather(V1, V2) :- parent(V1, V3), parent(V3, V2), male(V1).

Verification : 4/4 positifs couverts, 0/6 negatifs couverts


### Interpretation : la division du travail de l'ILP moderne

Une seule clause --- la cible, a renommage de variables pres --- couvre les 4
positifs sans toucher un negatif, et la boucle s'arrete au premier tour. Notez la
mecanique fine :

1. **Le negatif `grandfather(liz, eve)` fait tout le travail.** Sans lui, la clause
   plus courte `grandfather(X,Y) :- parent(X,Z), parent(Z,Y)` serait consistante et
   gagnerait au score (plus generale, moins de litteraux). C'est l'exemple negatif
   bien choisi --- une grand-mere --- qui force le litteral `male(X)`. La qualite
   des negatifs *est* le signal d'apprentissage.

2. **La clause bottom a fait passer l'espace de recherche de l'infini a quelques
   dizaines de candidats connectes.** C'est le pari de l'entailment inverse :
   troquer la completude theorique (toutes les clauses possibles) contre un espace
   sur-mesure pour *cet* exemple.

3. **FOIL et Progol se rejoignent au milieu.** FOIL descend depuis $\top$ guide
   par un gain statistique ; Progol enumere sous $\bot(e)$ guide par la
   compression. Sur ce probleme jouet ils trouvent la meme clause --- mais leur
   geometrie de recherche differe, et l'exercice 2 montre quand cela compte.


---

## 7. De Progol aux systemes modernes

| Systeme | Annee | Strategie | Apport |
|---------|-------|-----------|--------|
| FOIL (SL-4) | 1990 | top-down, gain d'information | simple, rapide, sensible aux pieges locaux |
| **Progol** | 1995 | **entailment inverse, A* sous $\bot(e)$** | **espace borne par exemple, declarations de modes** |
| Aleph | 2001 | reimplementation Prolog de Progol | le cheval de trait academique de l'ILP |
| Metagol | 2014 | meta-interpretation, regles d'ordre superieur | invente des predicats intermediaires |
| ILASP | 2015 | ASP (answer set programming) | apprend des programmes non-monotones |
| Popper | 2021 | learning from failures, contraintes | elagage du treillis par echecs generalises |

Les systemes recents (Metagol, Popper) repondent aux limites visibles ici : notre
recherche ne peut pas inventer de predicat intermediaire (`parent_de_parent`), ne
gere pas la recursion, et le bruit la fait echouer brutalement (exercice 3).

> **Note pratique** : Popper et Aleph s'appuient sur SWI-Prolog, absent des machines
> de TP --- leur integration est differee (Epic #1404). Le present notebook
> implemente les *mecanismes* (LGG, subsomption, bottom, compression) en Python pur,
> precisement pour que la mecanique reste visible.


---

## 8. Exercices

### Tableau recapitulatif

| Concept | Formule / idee | Implementation |
|---------|----------------|----------------|
| LGG (Plotkin) | anti-unification, $V_{s,t}$ par paire | `lgg_clause()` |
| Theta-subsomption | $\exists \theta : C\theta \subseteq D$ | `subsumes()` |
| Clause bottom | saturation bornee + variabilisation | `bottom_clause()` |
| Recherche Progol | sous-ensembles connectes de $\bot(e)$, $f = p - L$, $n = 0$ | `progol_search()` |
| Cover-set | retirer les positifs couverts, iterer | `progol_learn()` |


In [6]:
# Exercice 1 : Apprendre grandmother/2
# TODO etudiant : definissez les exemples positifs et negatifs de grandmother
# (la grand-mere) sur le MEME arbre genealogique, et faites-la apprendre par
# progol_learn. Quel negatif joue le role symetrique de grandfather(liz, eve) ?
# Indice : les grands-parents de l'arbre sont tom (x3), bob (jim) et liz (eve).
# Etape 1 : POS_GM (qui est grand-mere de qui ?) et NEG_GM (>= 5 exemples varies)
# Etape 2 : progol_learn(POS_GM, NEG_GM, FACTS, {"parent", "male", "female"})
# Etape 3 : verifiez la theorie apprise (female au lieu de male ?)

print("Exercice a completer")

Exercice a completer


L'exercice suivant teste le levier central de l'entailment inverse : la
profondeur de saturation. C'est elle qui decide si la cible est *dans* l'espace
de recherche.


In [7]:
# Exercice 2 : La profondeur de la clause bottom
# TODO etudiant : ajoutez la cible greatgrandfather/2 (arriere-grand-pere :
# trois sauts de parent + male) avec ses exemples, puis comparez l'apprentissage
# avec depth=1, 2 et 3 et max_len=4.
# Indice : les arriere-grands-peres de l'arbre : tom (jim via bob-pat, eve via
# liz-sue) ; bob n'en est pas un (jim n'a pas d'enfant).
# Etape 1 : POS_GGF, NEG_GGF (incluez un arriere-grand-parent femme si possible)
# Etape 2 : pour depth in (1, 2, 3) : taille de la clause bottom + resultat
# Etape 3 : a quelle profondeur minimale la cible devient-elle atteignable ?
#           Que se passe-t-il pour la taille de bottom si on monte encore ?

print("Exercice a completer")

Exercice a completer


L'exercice suivant introduit ce que tout systeme reel doit affronter : le
bruit. La contrainte de consistance stricte ($n = 0$) devient alors un piege.


In [8]:
# Exercice 3 : Tolerance au bruit
# TODO etudiant : ajoutez l'exemple POSITIF errone grandfather(liz, eve) (une
# erreur d'etiquetage !) a POS et observez progol_learn. Puis modifiez
# progol_search pour tolerer du bruit : remplacez la contrainte n = 0 par le
# score de compression de Progol f = p - n - L et retournez le meilleur f.
# Indice : copiez progol_search en progol_search_noisy, une ligne change dans
# le filtre et une dans le score.
# Etape 1 : run avec le positif bruite et la version stricte : que se passe-t-il ?
# Etape 2 : run avec progol_search_noisy : quelle clause gagne ? combien de
#           negatifs couvre-t-elle ?
# Etape 3 : la clause apprise avec bruit est-elle la cible ? Etait-ce previsible ?

print("Exercice a completer")

Exercice a completer


Dernier exercice : retour a la theorie. La subsomption est l'ordre *pratique*
de l'ILP, mais elle ne coincide pas exactement avec l'implication logique --- et
les clauses produites par LGG sont rarement *reduites*.


In [9]:
# Exercice 4 : Reduction de Plotkin
# TODO etudiant : une clause C est REDUITE si aucun sous-ensemble strict de son
# corps n'est equivalent a C sous subsomption (C' subsume C et C subsume C').
# Implementez reduce_clause(clause) qui retire un a un les litteraux redondants,
# et appliquez-la au LGG de la section 3.
# Indice : pour chaque litteral l du corps, testez si (head, body - {l}) et
# (head, body) se subsument mutuellement ; si oui, l est redondant.
# Etape 1 : reduce_clause + application au LGG g (combien de litteraux restent ?)
# Etape 2 : la clause reduite est-elle la cible ?
# Etape 3 (reflexion) : pourquoi la reduction complete est-elle NP-difficile,
#          alors que retirer UN litteral redondant est polynomial ?

print("Exercice a completer")

Exercice a completer


---

## Defi presentation (seance du 17 juin)

Modalite du cours : chaque groupe choisit **un exercice** de la serie, le prepare,
et le presente en seance. Resoudre l'exercice est le minimum ; ce qui distingue une
presentation qui maitrise le sujet, c'est la **question-twist** associee ci-dessous.
Elle fait partie integrante de la presentation attendue.

| Exercice | Question-twist a traiter en plus |
|----------|----------------------------------|
| **Ex. 1 (grandmother)** | Apprenez maintenant `grandparent/2` (sans contrainte de sexe) : pourquoi est-ce *plus facile* que grandfather, et qu'est-ce que cela dit du role des exemples negatifs comme porteurs du signal d'apprentissage ? |
| **Ex. 2 (profondeur de bottom)** | Estimez la croissance de $|\bot(e)|$ en fonction de la profondeur $i$ et du degre moyen du graphe de faits. A partir de quelle profondeur l'enumeration par sous-ensembles devient-elle intraitable, et comment les *declarations de modes* de Progol (types + recall) repoussent-elles cette limite ? |
| **Ex. 3 (bruit)** | Le score $f = p - n - L$ est un argument de longueur de description minimale (MDL). Construisez un jeu d'exemples ou ce score prefere une clause *fausse* a la cible, et expliquez quel terme du compromis (couverture, consistance, longueur) a ete trompe. |
| **Ex. 4 (reduction)** | Subsomption et implication ne coincident pas : la clause recursive $p(f(X)) \leftarrow p(X)$ implique $p(f(f(X))) \leftarrow p(X)$ sans la subsumer. Verifiez-le a la main, et expliquez pourquoi l'ILP travaille quand meme avec la subsomption (decidabilite, theoreme de Gottlob). |


---

## 9. Conclusion

### Recapitulatif

1. **Inverser la deduction** est le programme de l'ILP bottom-up : reconstruire la
   regle qui, avec la connaissance de fond, aurait permis de deduire l'exemple.

2. **Le LGG de Plotkin** generalise exactement le necessaire entre deux clauses,
   mais croit exponentiellement en cascade ; sa reduction est NP-difficile.

3. **La theta-subsomption** ($C\theta \subseteq D$) ordonne les clauses en un
   treillis ; decidable (NP-complete), elle approxime l'implication logique et
   structure toute la recherche.

4. **L'entailment inverse** (Progol) calcule pour chaque exemple une clause bottom
   $\bot(e)$ --- l'exemple sature et variabilise --- qui borne le treillis par le
   bas : la recherche enumere les sous-ensembles connectes de $\bot(e)$, scores par
   compression sous contrainte de consistance.

5. **Les negatifs portent le signal** : c'est `grandfather(liz, eve)` --- la
   grand-mere --- qui force le litteral `male(X)`. Sans bons negatifs, la clause la
   plus courte et la plus generale gagne toujours.

### Position dans la serie

| Methode | Direction | Espace de recherche | Garde-fou |
|---------|-----------|---------------------|-----------|
| FOIL (SL-4) | top-down | clauses de Horn, sans borne inferieure | gain d'information |
| **Progol (SL-9)** | **bottom-up** | **treillis $[\top, \bot(e)]$** | **compression + consistance** |
| AMIE (SL-6) | niveau par niveau | regles fermees sur KG | PCA confidence |
| L* (SL-8) | par requetes | DFA via table d'observation | minimalite garantie |

### Connexions

| Direction | Sujet | Lien |
|-----------|-------|------|
| Prerequis | Clauses de Horn, unification, FOIL, V/W | [SL-4](SL-4-InductiveLogicProgramming.ipynb) |
| Regles sur graphes | AMIE, PCA | [SL-6](SL-6-KnowledgeGraphs-ILP.ipynb) |
| Capstone | pipeline LLM -> oracle -> ILP -> KG | [SL-10](SL-10-Capstone-NeuroSymbolic.ipynb) |


---

## Ressources

- G. Plotkin, *A Note on Inductive Generalization*, Machine Intelligence 5, 1970
- S. Muggleton, *Inverse Entailment and Progol*, New Generation Computing 13, 1995
- S. Muggleton & L. De Raedt, *Inductive Logic Programming: Theory and Methods*, J. Logic Programming 19-20, 1994
- G. Gottlob, *Subsumption and Implication*, Information Processing Letters 24(2), 1987
- A. Cropper & R. Morel, *Learning Programs by Learning from Failures* (Popper), Machine Learning 110, 2021
- [Aleph](https://www.cs.ox.ac.uk/activities/programinduction/Aleph/aleph.html) --- le Progol academique de reference
- Russell & Norvig, *AI: A Modern Approach*, 3e ed., section 19.5

---

**Retour** : [Index SymbolicLearning](./README.md) | [<< SL-8](SL-8-ActiveAutomataLearning.ipynb) | [SL-10 >>](SL-10-Capstone-NeuroSymbolic.ipynb)
